![Logo](img/logoitqv1.jpg) 

# Modificación del modelo: Neurona de McCulloch y Pitts

![logo](img/python_logo.png)

[Génesis Escobar](https://github.com/GenesisEsc/machinelearning-2.git)

**¿Por qué hacemos esto?**

Necesitamos que la neurona aprenda exclusivamente de las 5 características que más impactan en el diagnóstico de cáncer de mama. Además, como la Neurona de McCulloch-Pitts solo entiende valores booleanos (0 y 1), debemos asegurarnos de que la información se transforme correctamente antes de entrenar y exportar el modelo.

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score

# Cargar el dataset

In [3]:
cancer_mama = load_breast_cancer()
X = pd.DataFrame(cancer_mama.data, columns=cancer_mama.feature_names)
Y = cancer_mama.target 

# SELECCIÓN DE CARACTERÍSTICAS

Usamos SelectKBest para extraer matemáticamente las 5 columnas más importantes

In [4]:
selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X, Y)

# Guardamos los nombres de esas 5 columnas para saber cuáles poner en el HTML
columnas_seleccionadas = X.columns[selector.get_support(indices=True)].tolist()
print("Los 5 parámetros más importantes son:", columnas_seleccionadas)

X_reduced = pd.DataFrame(X_selected, columns=columnas_seleccionadas)

Los 5 parámetros más importantes son: ['mean perimeter', 'mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']


# BINARIZACIÓN

La neurona MP requiere ceros y unos. Una forma sencilla es asignar 1 si el valor del tumor supera el promedio histórico, y 0 si está por debajo.

In [5]:
X_bin = (X_reduced > X_reduced.mean()).astype(int)

# DIVISIÓN DE DATOS

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_bin, Y, test_size=0.25, stratify=Y, random_state=42)
print("Tamaño de entrenamiento ", len(X_train))
print("Tamaño de pruebas ", len(X_test))

Tamaño de entrenamiento  426
Tamaño de pruebas  143


# DEFINICIÓN Y ENTRENAMIENTO DE LA NEURONA

In [10]:
class MpNeurona:
    def __init__(self):
        self.threshold = None
    def model(self, x):
        # INVERTIMOS LA LÓGICA AQUÍ
        # Si la suma de anomalías es menor o igual al umbral, es Benigno (1)
        return sum(x) <= self.threshold
    def predict(self, X):
        return np.array([self.model(x) for x in X])
    def fit(self, X, Y):
        accuracy = {}
        for th in range(X.shape[1] + 1):
            self.threshold = th
            Y_pred = self.predict(X)
            accuracy[th] = accuracy_score(Y_pred, Y)
        self.threshold = max(accuracy, key=accuracy.get)

# Entrenamos el nuevo modelo de 5 variables
np_neurona = MpNeurona()
np_neurona.fit(X_train.to_numpy(), y_train)
print("Threshold óptimo encontrado:", np_neurona.threshold)

Threshold óptimo encontrado: 2


# EXPORTAR EL MODELO

Guardamos la neurona entrenada en un archivo físico para que la web lo pueda leer.

In [11]:
with open('mp_neurona_optimizada.pkl', 'wb') as file:
    pickle.dump(np_neurona, file)